In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("CTD_same_question_response_final.pkl", "rb") as f:
    CTD_same_question_response = pickle.load(f)


In [5]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTarget", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    # if model=="OpenTarget":
    #     dct_result[model] = Opentarget_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]

    if model=="ctd":
        dct_result[model] = CTD_same_question_response[model]

    # if model=="hcdt":
    #     dct_result[model] = HCDT_same_question_response[model]


##### Verify if only valid columns are present

In [6]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 309
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [7]:
dct_jaccard = dict()

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            if not set(df.columns).issubset(allowed_cols):
                continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: ctd


##### Associate id with all gene entry

In [8]:
# Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
df_gene = resolve_ensembl_ids_with_fallback(
    list(question_entities["gene_name"]),
    use_opentargets_fallback=True,
)
df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
df_gene


2026-02-27 17:43:28,757 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:43:28,758 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:43:28,759 [INFO] querying 1-1000 ...
2026-02-27 17:43:31,202 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 17:43:32,209 [INFO] querying 1001-2000 ...
2026-02-27 17:43:33,560 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 17:43:34,566 [INFO] querying 2001-2047 ...
2026-02-27 17:43:34,876 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 17:43:35,879 [INFO] Finished.
2026-02-27 17:43:36,011 [WARNING] 5 input query terms found dup hits:	[('CMAHP', 2), ('CTSLP8', 2), ('MNX1-AS1', 2), ('SNHG14', 2), ('VNN3P', 2)]
2026-02-27 17:43:36,012 [WARNING] 29 input query terms found no hit:	['ABCB1A', 'ABCB1B', 'ABL', 'ARSE', 'ATP6', 'ATP7', 'COX1', 'COX3', 'CT

[warn] Could not resolve 'ABCB1A' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'ABCB1B' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'ATP7' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'CTR1B' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'CTR1C' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'CYP2B1' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'DISC2' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'H1F10' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'MIR124A-3' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'MIR18

,gene_name,gene_id,source
0,A1BG,ENSG00000121410,mygene
1,A2M,ENSG00000175899,mygene
2,AARS1,ENSG00000090861,mygene
3,ABAT,ENSG00000183044,mygene
4,ABCA2,ENSG00000107331,mygene
...,...,...,...
2042,ZNF565,ENSG00000196357,mygene
2043,ZNF765,ENSG00000196417,mygene
2044,ZNF804A,ENSG00000170396,mygene
2045,ZNF816,ENSG00000180257,mygene


In [9]:
df_gene[df_gene.isna().any(axis=1)]

,gene_name,gene_id,source
9,ABCB1A,NaN,None
10,ABCB1B,NaN,None
182,ATP7,NaN,None
462,CTR1B,NaN,None
463,CTR1C,NaN,None
498,CYP2B1,NaN,None
562,DISC2,NaN,None
847,H1F10,NaN,None
1180,MIR124A-3,NaN,None
1188,MIR181B-1,NaN,None


##### Associate id with all drug entry

In [10]:
# Resolve unique drug surface forms to IDs.
# This keeps original raw names as rows and writes mapped ID + source.
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
df_drug


2026-02-27 17:45:02,983 [INFO] [LLM] Total 573 names split into 8 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:45:02,987 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,032 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,036 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,039 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,040 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,040 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,041 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:03,041 [INFO] [LLM] Batch size: 13


[map][drug][OpenTargets] raw='10-nitro-oleic acid' -> id='CHEMBL561371'
[map][drug][OpenTargets] raw='2-Methoxyestradiol' -> id='CHEMBL299613'
[map][drug][OpenTargets] raw='3-methyladenine' -> id='CHEMBL292268'
[map][drug][OpenTargets] raw='Acetaminophen' -> id='CHEMBL112'
[map][drug][OpenTargets] raw='Acetazolamide' -> id='CHEMBL20'
[map][drug][OpenTargets] raw='Acetic Acid' -> id='CHEMBL539'
[map][drug][OpenTargets] raw='Acetylcysteine' -> id='CHEMBL600'
[map][drug][OpenTargets] raw='Adenosine Triphosphate' -> id='CHEMBL14249'
[map][drug][OpenTargets] raw='Ado-Trastuzumab Emtansine' -> id='CHEMBL1743082'
[map][drug][OpenTargets] raw='Adrenocorticotropic Hormone' -> id='CHEMBL1201610'
[map][drug][OpenTargets] raw='Afatinib' -> id='CHEMBL1173655'
[map][drug][OpenTargets] raw='Alitretinoin' -> id='CHEMBL705'
[map][drug][OpenTargets] raw='Allopurinol' -> id='CHEMBL1467'
[map][drug][OpenTargets] raw='Alprenolol' -> id='CHEMBL266195'
[map][drug][OpenTargets] raw='Aluminum Oxide' -> id='CHE

2026-02-27 17:45:04,890 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:04,918 [INFO] [LLM] Batch done in 1.88s
2026-02-27 17:45:04,971 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:04,974 [INFO] [LLM] Batch done in 1.94s
2026-02-27 17:45:06,578 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:06,581 [INFO] [LLM] Batch done in 3.54s
2026-02-27 17:45:07,769 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:07,771 [INFO] [LLM] Batch done in 4.73s
2026-02-27 17:45:08,229 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:08,231 [INFO] [LLM] Batch done in 5.19s
2026-02-27 17:45:08,716 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:08

[map][drug][Llama] raw='1,25-dihydroxyvitamin D' -> canonical='CALCITRIOL' -> id='CHEMBL846'
[map][drug][Llama] raw='1-Methyl-3-isobutylxanthine' -> canonical='PENTOXIFYLLINE' -> id='CHEMBL628'
[map][drug][Llama] raw='MK 2206' -> canonical='MK-2206' -> id='CHEMBL1079175'
[map][drug][Llama] raw='beta-hexachlorocyclohexane' -> canonical='LINDANE' -> id='CHEMBL15891'
[map][drug][Llama] raw='flupenthixol decanoate' -> canonical='FLUPENTIXOL' -> id='CHEMBL6066864'
[map][drug][Llama] raw='fluphenazine depot' -> canonical='FLUPHENAZINE' -> id='CHEMBL726'
[map][drug][Llama] raw='insulin, Asp(B10)-' -> canonical='INSULIN ASPART' -> id='CHEMBL1201496'


,drug_name,drug_id,source
0,((1-ethylpyrrolidin-2-yl) methyl)-4-hydroxy-7-...,None,None
1,(+)-JQ1 compound,None,None
2,"(3R)-((2,3-dihydro-5-methyl-3-((4-morpholinyl)...",None,None
3,"(6,7-dimethoxyquinolin-4-yl)(5-methoxy-2,2-dim...",None,None
4,"1,1-bis(3'-indolyl)-1-(4-hydroxyphenyl)methane",None,None
...,...,...,...
1142,zafirlukast,CHEMBL603,OpenTargets
1143,zhenwu decoction,None,None
1144,zileuton,CHEMBL93,OpenTargets
1145,zinc chloride,CHEMBL1200679,OpenTargets


In [11]:
df_drug["source"].value_counts()

source
OpenTargets    574
Llama            7
Name: count, dtype: int64

In [12]:
# df_drug[df_drug["source"]=="Llama"]

In [13]:
df_drug[df_drug.isna().any(axis=1)]

,drug_name,drug_id,source
0,((1-ethylpyrrolidin-2-yl) methyl)-4-hydroxy-7-...,None,None
1,(+)-JQ1 compound,None,None
2,"(3R)-((2,3-dihydro-5-methyl-3-((4-morpholinyl)...",None,None
3,"(6,7-dimethoxyquinolin-4-yl)(5-methoxy-2,2-dim...",None,None
4,"1,1-bis(3'-indolyl)-1-(4-hydroxyphenyl)methane",None,None
...,...,...,...
1133,tyrphostin AG825,None,None
1136,vanadyl sulfate,None,None
1140,xianglian pill,None,None
1141,yufengningxin,None,None


##### Associate id with all disease entry

In [14]:
# Resolve unique disease surface forms to IDs.
# Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
df_disease


2026-02-27 17:45:12,755 [INFO] [LLM] Total 236 names split into 3 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:45:12,756 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:12,757 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:12,759 [INFO] [LLM] Batch size: 76


[map][disease][OpenTargets] raw='AMYOTROPHIC LATERAL SCLEROSIS 15 WITH OR WITHOUT FRONTOTEMPORAL DEMENTIA' -> id='MONDO_0010459'
[map][disease][OpenTargets] raw='AMYOTROPHIC LATERAL SCLEROSIS 18' -> id='MONDO_0013891'
[map][disease][OpenTargets] raw='AMYOTROPHIC LATERAL SCLEROSIS 20' -> id='MONDO_0014181'
[map][disease][OpenTargets] raw='AMYOTROPHIC LATERAL SCLEROSIS-PARKINSONISM/DEMENTIA COMPLEX 1' -> id='MONDO_0007104'
[map][disease][OpenTargets] raw='ANTLEY-BIXLER SYNDROME WITHOUT GENITAL ANOMALIES OR DISORDERED STEROIDOGENESIS' -> id='MONDO_0020667'
[map][disease][OpenTargets] raw='Abdominal Pain' -> id='HP_0002027'
[map][disease][OpenTargets] raw='Acquired Hyperostosis Syndrome' -> id='EFO_1001164'
[map][disease][OpenTargets] raw='Acrocephalosyndactylia' -> id='MONDO_0019796'
[map][disease][OpenTargets] raw='Acute Kidney Injury' -> id='MONDO_0002492'
[map][disease][OpenTargets] raw='Acute erythroleukemia' -> id='EFO_0000218'
[map][disease][OpenTargets] raw='Adenocarcinoma' -> id='

2026-02-27 17:45:15,510 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:15,513 [INFO] [LLM] Batch done in 2.75s
2026-02-27 17:45:15,590 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:15,593 [INFO] [LLM] Batch done in 2.84s
2026-02-27 17:45:17,676 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:17,679 [INFO] [LLM] Batch done in 4.92s
2026-02-27 17:45:17,680 [INFO] [LLM] 'AMYOTROPHIC LATERAL SCLEROSIS 12 WITH OR WITHOUT FRONTOTEMPORAL DEMENTIA' → 'amyotrophic lateral sclerosis'
2026-02-27 17:45:17,681 [INFO] [LLM] 'AMYOTROPHIC LATERAL SCLEROSIS 16, JUVENILE' → 'amyotrophic lateral sclerosis'
2026-02-27 17:45:17,682 [INFO] [LLM] 'AMYOTROPHIC LATERAL SCLEROSIS 19' → 'amyotrophic lateral sclerosis'
2026-02-27 17:45:17,682 [INFO] [LLM] 'Abnormalities, Multiple' → None
2026-02-27 17:45:17,683 [INFO] [LLM] 'Abort

[map][disease][Llama] raw='AMYOTROPHIC LATERAL SCLEROSIS 12 WITH OR WITHOUT FRONTOTEMPORAL DEMENTIA' -> canonical='amyotrophic lateral sclerosis' -> id='MONDO_0004976'
[map][disease][Llama] raw='AMYOTROPHIC LATERAL SCLEROSIS 16, JUVENILE' -> canonical='amyotrophic lateral sclerosis' -> id='MONDO_0004976'
[map][disease][Llama] raw='AMYOTROPHIC LATERAL SCLEROSIS 19' -> canonical='amyotrophic lateral sclerosis' -> id='MONDO_0004976'
[map][disease][Llama] raw='Abortion, Spontaneous' -> canonical='spontaneous abortion' -> id='EFO_1001255'
[map][disease][Llama] raw='Acidosis, Lactic' -> canonical='lactic acidosis' -> id='EFO_1000036'
[map][disease][Llama] raw='Adenocarcinoma, Bronchiolo-Alveolar' -> canonical='adenocarcinoma' -> id='EFO_0000228'
[map][disease][Llama] raw='Adenocarcinoma, Clear Cell' -> canonical='adenocarcinoma' -> id='EFO_0000228'
[map][disease][Llama] raw='Adenocarcinoma, Follicular' -> canonical='adenocarcinoma' -> id='EFO_0000228'
[map][disease][Llama] raw='Adrenocortica

,disease_name,disease_id,source
0,AMYOTROPHIC LATERAL SCLEROSIS 12 WITH OR WITHO...,MONDO_0004976,Llama
1,AMYOTROPHIC LATERAL SCLEROSIS 15 WITH OR WITHO...,MONDO_0010459,OpenTargets
2,"AMYOTROPHIC LATERAL SCLEROSIS 16, JUVENILE",MONDO_0004976,Llama
3,AMYOTROPHIC LATERAL SCLEROSIS 18,MONDO_0013891,OpenTargets
4,AMYOTROPHIC LATERAL SCLEROSIS 19,MONDO_0004976,Llama
...,...,...,...
489,Vulvar Lichen Sclerosus,EFO_1000623,OpenTargets
490,Weight Gain,HP_0004324,OpenTargets
491,Weight Loss,HP_0001824,OpenTargets
492,Williams Syndrome,MONDO_0008678,OpenTargets


In [15]:
df_disease["source"].value_counts()

source
OpenTargets    258
Llama          138
Name: count, dtype: int64

In [16]:
df_disease[df_disease.isna().any(axis=1)]

,disease_name,disease_id,source
9,"Abnormalities, Multiple",None,None
56,"Aortic Valve, Calcification of",None,None
57,Arsenic Poisoning,None,None
58,"Arthritis, Experimental",None,None
74,"Bone Diseases, Developmental",None,None
...,...,...,...
475,"Tuberculosis, Spinal",None,None
480,Urologic Neoplasms,None,None
484,"Vasculitis, Leukocytoclastic, Cutaneous",None,None
485,"Ventricular Dysfunction, Left",None,None


In [17]:
# ------------------------------------------------------------------
# Disease canonical audit (OpenTargets-strict IDs only)
# ------------------------------------------------------------------
# IMPORTANT:
# - This cell runs an additional Llama canonicalization pass over unresolved rows.
# - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# - To avoid changing the baseline resolver output, writeback is OFF by default.
APPLY_AUDIT_MAPPINGS = False

# 1) Rows unresolved after primary resolver
unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
unresolved_raw_names = list(
    dict.fromkeys(
        str(name).strip()
        for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
        if str(name).strip()
    )
)

# 2) Llama canonicalization suggestions
disease_input_map = {name: None for name in unresolved_raw_names}
canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# 3) OpenTargets strict verification (score==1 only)
canonical_terms_norm = list(
    dict.fromkeys(
        utility._normalize_disease_term(v)
        for v in canonicalisation_map.values()
        if isinstance(v, str) and v.strip()
    )
)
canonical_to_id = utility.resolve_diseases_opentargets_bulk(
    canonical_terms_norm,
    strict_score_one=True,
)

# 4) Build persistent audit table
audit_rows = []
applied_count = 0
candidate_count = 0

for raw_disease_name in unresolved_raw_names:
    canonical_name = canonicalisation_map.get(raw_disease_name)
    if isinstance(canonical_name, str) and canonical_name.strip():
        canonical_name = canonical_name.strip()
        canonical_norm = utility._normalize_disease_term(canonical_name)
        resolved_id = canonical_to_id.get(canonical_norm)
    else:
        canonical_name = None
        canonical_norm = None
        resolved_id = None

    can_apply = bool(resolved_id)
    if can_apply:
        candidate_count += 1
        print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

    if can_apply and APPLY_AUDIT_MAPPINGS:
        mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
        df_disease.loc[mask, "disease_id"] = resolved_id
        df_disease.loc[mask, "source"] = "Llama"
        applied_count += 1

    status = (
        "mappable_ot_exact" if can_apply else
        ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
    )
    audit_rows.append({
        "raw_disease_name": raw_disease_name,
        "canonical_name": canonical_name,
        "canonical_norm": canonical_norm,
        "resolved_disease_id": resolved_id,
        "status": status,
        "can_apply": can_apply,
        "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
    })

disease_canonical_audit_df = pd.DataFrame(audit_rows)
if not disease_canonical_audit_df.empty:
    disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
        ["status", "raw_disease_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
if not disease_canonical_audit_df.empty:
    print("[summary][disease][audit] status counts:")
    print(disease_canonical_audit_df["status"].value_counts(dropna=False))

disease_canonical_audit_df


2026-02-27 17:45:19,479 [INFO] [LLM] Total 98 names split into 2 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:45:19,480 [INFO] [LLM] Batch size: 80
2026-02-27 17:45:19,481 [INFO] [LLM] Batch size: 18
2026-02-27 17:45:20,162 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:20,165 [INFO] [LLM] Batch done in 0.68s
2026-02-27 17:45:21,516 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:45:21,518 [INFO] [LLM] Batch done in 2.04s
2026-02-27 17:45:21,519 [INFO] [LLM] 'Abnormalities, Multiple' → None
2026-02-27 17:45:21,520 [INFO] [LLM] 'Aortic Valve, Calcification of' → None
2026-02-27 17:45:21,520 [INFO] [LLM] 'Arsenic Poisoning' → 'arsenic poisoning'
2026-02-27 17:45:21,521 [INFO] [LLM] 'Arthritis, Experimental' → None
2026-02-27 17:45:21,521 [INFO] [LLM] 'Bone Diseases, Developmental' → None
2026-02-27 17:45:21,521 [INFO] 

[audit][disease][Llama->OT] raw='Brain Injuries' -> canonical='brain injury' -> id='MONDO_0043510'
[audit][disease][Llama->OT] raw='SEIZURES, BENIGN FAMILIAL INFANTILE, 5' -> canonical='benign familial infantile epilepsy' -> id='MONDO_0017615'
[audit][disease][Llama->OT] raw='Schizophrenia Spectrum and Other Psychotic Disorders' -> canonical='schizophrenia' -> id='MONDO_0005090'
[summary][disease][audit] OT-mappable candidates: 3
[summary][disease][audit] applied to df_disease: 0 (APPLY_AUDIT_MAPPINGS=False)
[summary][disease][audit] status counts:
status
no_canonical           93
mappable_ot_exact       3
canonical_not_in_ot     2
Name: count, dtype: int64


,raw_disease_name,canonical_name,canonical_norm,resolved_disease_id,status,can_apply,applied
0,"Tuberculosis, Bovine","Tuberculosis, Bovine","tuberculosis, bovine",None,canonical_not_in_ot,False,False
1,"Tuberculosis, Spinal","tuberculosis, spinal","tuberculosis, spinal",None,canonical_not_in_ot,False,False
2,Brain Injuries,brain injury,brain injury,MONDO_0043510,mappable_ot_exact,True,False
3,"SEIZURES, BENIGN FAMILIAL INFANTILE, 5",benign familial infantile epilepsy,benign familial infantile epilepsy,MONDO_0017615,mappable_ot_exact,True,False
4,Schizophrenia Spectrum and Other Psychotic Dis...,schizophrenia,schizophrenia,MONDO_0005090,mappable_ot_exact,True,False
...,...,...,...,...,...,...,...
93,"Tuberculosis, Osteoarticular",None,None,None,no_canonical,False,False
94,Urologic Neoplasms,None,None,None,no_canonical,False,False
95,"Vasculitis, Leukocytoclastic, Cutaneous",None,None,None,no_canonical,False,False
96,"Ventricular Dysfunction, Left",None,None,None,no_canonical,False,False


In [18]:
# disease_canonical_audit_df

In [19]:
df_disease[df_disease["source"]=='Llama']

,disease_name,disease_id,source
0,AMYOTROPHIC LATERAL SCLEROSIS 12 WITH OR WITHO...,MONDO_0004976,Llama
2,"AMYOTROPHIC LATERAL SCLEROSIS 16, JUVENILE",MONDO_0004976,Llama
4,AMYOTROPHIC LATERAL SCLEROSIS 19,MONDO_0004976,Llama
10,"Abortion, Spontaneous",EFO_1001255,Llama
11,"Acidosis, Lactic",EFO_1000036,Llama
...,...,...,...
470,"Tuberculosis, Multidrug-Resistant",EFO_0007381,Llama
472,"Tuberculosis, Pleural",EFO_0007446,Llama
473,"Tuberculosis, Pulmonary",EFO_1000049,Llama
474,"Tuberculosis, Renal",EFO_0007463,Llama


In [20]:
df_disease["source"].value_counts()

source
OpenTargets    258
Llama          138
Name: count, dtype: int64

In [21]:
df_disease

,disease_name,disease_id,source
0,AMYOTROPHIC LATERAL SCLEROSIS 12 WITH OR WITHO...,MONDO_0004976,Llama
1,AMYOTROPHIC LATERAL SCLEROSIS 15 WITH OR WITHO...,MONDO_0010459,OpenTargets
2,"AMYOTROPHIC LATERAL SCLEROSIS 16, JUVENILE",MONDO_0004976,Llama
3,AMYOTROPHIC LATERAL SCLEROSIS 18,MONDO_0013891,OpenTargets
4,AMYOTROPHIC LATERAL SCLEROSIS 19,MONDO_0004976,Llama
...,...,...,...
489,Vulvar Lichen Sclerosus,EFO_1000623,OpenTargets
490,Weight Gain,HP_0004324,OpenTargets
491,Weight Loss,HP_0001824,OpenTargets
492,Williams Syndrome,MONDO_0008678,OpenTargets


In [22]:
# df_disease[df_disease["source"]=="Llama"]

In [23]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()

                # print(work_df.head())

                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)


Working for model: ctd
'Which diseases are associated with BRAF?': J=1.0000, |∩|=34, |∪|=34, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.9904, |∩|=103, |∪|=104, runs=5
'Which diseases are associated with CDK4?': J=1.0000, |∩|=9, |∪|=9, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=80, |∪|=80, runs=5
'Which genes are associated with ovarian cancer?': J=0.1988, |∩|=32, |∪|=161, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=24, |∪|=24, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
Not enough valid runs for 'Which genes are associated with fever (pyrexia)?' (0/0). Skipping.
Not enough valid runs for 'Which diseases are treated with bevacizumab?' (0/0). Skipping.
'What are the known targets of cabozantinib?': J=1.0000, |∩|=3, |∪|=3, runs=5
Not enough valid r

In [24]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: ctd
'Which diseases are associated with BRAF?': J=1.0000, |∩|=34, |∪|=34, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.9904, |∩|=103, |∪|=104, runs=5
'Which diseases are associated with CDK4?': J=1.0000, |∩|=9, |∪|=9, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=80, |∪|=80, runs=5
'Which genes are associated with ovarian cancer?': J=0.1988, |∩|=32, |∪|=161, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=24, |∪|=24, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
Not enough valid runs for 'Which genes are associated with fever (pyrexia)?' (0/0). Skipping.
Not enough valid runs for 'Which diseases are treated with bevacizumab?' (0/0). Skipping.
'What are the known targets of cabozantinib?': J=1.0000, |∩|=3, |∪|=3, runs=5
Not enough valid r

,model,question,jaccard,n_valid_runs,intersection_size,union_size
0,ctd,Which diseases are associated with BRAF?,1.0,5,34,34
2,ctd,Which diseases are associated with CDK4?,1.0,5,9,9
3,ctd,Which drugs are used to treat rheumatoid arthr...,1.0,5,80,80
5,ctd,What are the known targets of regorafenib?,1.0,5,24,24
6,ctd,What are the known targets of cabozantinib?,1.0,5,3,3
7,ctd,Which diseases are associated with the NPM1 gene?,1.0,5,8,8
8,ctd,Which drugs are used to treat migraine?,1.0,5,119,119
9,ctd,What drugs are used to treat rickets?,1.0,5,7,7
11,ctd,Which genes are associated with Alzheimer’s di...,1.0,5,105,105
12,ctd,Which genes are associated with renal cell car...,1.0,5,134,134


In [25]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("CTD_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
